In [2]:
import pandas as pd
import ast
from sklearn.model_selection import GroupKFold
import numpy as np
import os
import re

In [6]:
df = pd.read_csv('./data/annotated/full_ner/all_annotations_minimal_fixed_multi.csv')

In [8]:
len(set(df['doc_id']))

1450

In [10]:
df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['ner_tags'] = df['ner_tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['annotations_tuples'] = df['annotations_tuples'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df['tokens_length'] = df['tokens'].apply(len)
df['ner_tags_length'] = df['ner_tags'].apply(len)
df['length_match'] = df['tokens_length'] == df['ner_tags_length']
df.head()

,doc_id,Text,tokens,ner_tags,annotations_tuples,tokens_length,ner_tags_length,length_match
0,My_pdf325new_method,\r\n2 | MATERIALS AND METHODS\r\n2.1 | Ethics ...,"[\r\n, 2, |, MATERIALS, AND, METHODS, \r\n, 2....","[O, O, O, O, O, O, O, O, O, O, O, O, B-welfare...","[(53, 189, welfare, Each procedure was conduct...",4016,4016,True
1,My_pdf41new_method,\r\nMaterials and Methods\r\nAntibodies and re...,"[\r\n, Materials, and, Methods, \r\n, Antibodi...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(867, 1037, weight, Adult male Sprague Dawley...",4265,4265,True
2,My_pdf746_method,\r\nMethods\r\nEthical approval\r\nAll procedu...,"[\r\n, Methods, \r\n, Ethical, approval, \r\n,...","[O, O, O, O, O, O, B-welfare, I-welfare, I-wel...","[(29, 212, welfare, All procedures and protoco...",1573,1573,True
3,My_pdf746_title^abstract,Effect of monoamine reuptake inhibition and al...,"[Effect, of, monoamine, reuptake, inhibition, ...","[O, O, B-therapy-drug, I-therapy-drug, I-thera...","[(10, 39, therapy-drug, monoamine reuptake inh...",365,365,True
4,My_pdf183new_method,\r\n3.1. Method performance evaluation\r\nThe ...,"[\r\n, 3.1, ., Method, performance, evaluation...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(1719, 1723, therapy-other, MENK), (1768, 177...",2071,2071,True


## Split text into sentences

In [13]:
import re

def should_merge(prev, next_):
    """
    Determines whether to merge the current sentence-ending token with the next token,
    based on citation patterns, abbreviations, numbers, and common edge cases in scientific writing.
    
    Parameters:
    - prev (str): The previous token (typically a sentence-ending punctuation like ".")
    - next_ (str): The next token (typically the first token of the potential next sentence)
    
    Returns:
    - bool: True if the tokens should be merged (i.e., no sentence split), False otherwise
    """
    # Handle split medical abbreviations like 'i.c.v' + '.'
    # Case when the abbreviation and the period are in two separate tokens
    if prev.strip().lower() in ['i.c.v', 'i.p', 'i.v', 'i.m', 's.c', 'i.t', 'p.o',
                                'inh', 'no', 'al', 'fig', 'figs', 'table', 'ref']:
        return True


    # Rule: merge if both prev and next are lowercase (e.g., abbrev. continuation)
    if prev.islower() and next_.islower():
        return True

    # Previous token patterns (e.g., abbreviations or known "no split" cases)
    prev_patterns = [
        r'^(?:i\.c\.v|i\.p|i\.v|i\.m|s\.c|i\.t|p\.o)\.$',              # Administration routes
        r'\b(?:Fig|Figs|Table|Ref|Inc|inc|c|p|g|no|No|vs|sp|spp)\.$',  # Abbreviations
        r'et al\.$',                                                   # "et al."
        r'(?:\s|\.)[A-Z]\.$',                                          # Initials
        r'\d+\.\d+\)$'                                                 # Closing parenthesis
    ]
    
    # Next token patterns (e.g., signs the sentence should continue)
    next_patterns = [
        r'^,$',                   # Comma (e.g., in citations)
        r'^\)$',                  # Closing parenthesis
        r'^[a-z]',                # Lowercase continuation
        r'^\(',                   # Opening parenthesis
        r'^Mb\b',                 # Scientific abbreviations
        r'^M\b',
        r'^\d',                   # Starts with digit (e.g., year, dose)
        r'^al$',                  # Part of "et al"
        r'^al\.$',
        r'^et$',                  # "et"
        r'^\w+\,?\s?\d{4}[a-z]?$',# Citation year like "2000b"
        r'^\r\n$',                # Newline token
        #r'^[A-Z]{2,}[0-9]*$',     # License codes like SCXK
    ]
    
    for pattern in prev_patterns:
        if re.fullmatch(pattern, prev):
            return True

    for pattern in next_patterns:
        if re.fullmatch(pattern, next_):
            return True

    return False

def split_into_sentences(tokens, ner_tags):
    sentences = []
    current_tokens = []
    current_tags = []

    for i in range(len(tokens)):
        tok = tokens[i]
        tag = ner_tags[i]
        current_tokens.append(tok)
        current_tags.append(tag)

        # Check for sentence-ending punctuation
        if tok in [".", "!", "?", "\n", "\r\n"]:
            prev_tok = tokens[i - 1] if i > 0 else ""
            next_tok = tokens[i + 1] if i + 1 < len(tokens) else ""

            # ✅ SPECIAL CASE: newline followed by lowercase = merge
            if tok in ['\n', '\r\n'] and next_tok and next_tok[0].islower():
                continue  # do not split

            # Existing should_merge logic
            if should_merge(prev_tok, next_tok):
                continue

            # Commit the sentence split
            sentences.append((current_tokens, current_tags))
            current_tokens = []
            current_tags = []

    # Add final remaining chunk
    if current_tokens:
        sentences.append((current_tokens, current_tags))

    return sentences


In [15]:
# === MAIN PROCESSING ===

# Assuming `df` is your input DataFrame with 'tokens' and 'ner_tags' columns
new_rows = []

for _, row in df.iterrows():
    tokens = row['tokens']
    tags = row['ner_tags']

    # Sentence split
    sentences = split_into_sentences(tokens, tags)

    for idx, (sent_tokens, sent_tags) in enumerate(sentences):
        new_rows.append({
            'doc_id': row['doc_id'],
            'sentence_id': idx,
            'tokens': sent_tokens,
            'ner_tags': sent_tags,
            'sent_txt': ' '.join(sent_tokens)
        })

# Build the initial sentence DataFrame
sentence_df = pd.DataFrame(new_rows)

In [16]:
sentence_df.to_csv("data/debug_regex/./data/sentence_split_v2.csv")

In [19]:
def extract_entity_types(tags):
    entity_types = set()
    for tag in tags:
        if tag != "O":
            # Strip B- or I- prefix to get the entity type
            types_list = tag.split("|")
            for types in types_list:
                entity_type = types.split("-", 1)[-1]
                entity_types.add(entity_type)
    return list(entity_types)  # Or use sorted(entity_types) if you want consistency

# Apply this function to your sentence-level dataframe
sentence_df['entity_types'] = sentence_df['ner_tags'].apply(extract_entity_types)
sentence_df

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types
0,My_pdf325new_method,0,"[\r\n, 2, |, MATERIALS, AND, METHODS, \r\n]","[O, O, O, O, O, O, O]",\r\n 2 | MATERIALS AND METHODS \r\n,[]
1,My_pdf325new_method,1,"[2.1, |, Ethics, statement, \r\n]","[O, O, O, O, O]",2.1 | Ethics statement \r\n,[]
2,My_pdf325new_method,2,"[Each, procedure, was, conducted, in, accordan...","[B-welfare, I-welfare, I-welfare, I-welfare, I...",Each procedure was conducted in accordance wit...,[welfare]
3,My_pdf325new_method,3,"[All, efforts, were, made, to, minimize, the, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",All efforts were made to minimize the number o...,[]
4,My_pdf325new_method,4,"[2.2, |, Animal, grouping, and, model, establi...","[O, O, O, O, O, O, O, O]",2.2 | Animal grouping and model establishment ...,[]
...,...,...,...,...,...,...
66103,My_pdf96new_method,123,"[If, P, <, .]","[O, O, O, O]",If P < .,[]
66104,My_pdf96new_method,124,"[05, ,, F, -, ratios, were, calculated, using,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]","05 , F - ratios were calculated using the Gree...",[]
66105,My_pdf96new_method,125,"[The, accepted, level, of, significance, for, ...","[O, O, O, O, O, O, O, O, O, O, O, O]",The accepted level of significance for the tes...,[]
66106,My_pdf96new_method,126,"[05, .]","[O, O]",05 .,[]


In [20]:
sentence_df.to_csv("data/debug_regex/./data/sentence_split.csv", index=False)

In [23]:
sentence_df = pd.read_csv("./data/sentence_split.csv")

In [24]:
entity_labels = ['age', 'welfare', 'blinding', 'randomization', 'arrive', 'power']
counts = {}

for label in entity_labels:
    sentence_df[label] = sentence_df['entity_types'].apply(lambda ents: 1 if label in ents else 0)
    counts[label] = sentence_df[label].sum()

print(counts)

{'age': 488, 'welfare': 706, 'blinding': 392, 'randomization': 500, 'arrive': 14, 'power': 25}


In [27]:
sentence_df.head()

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,welfare,blinding,randomization,arrive,power
0,My_pdf325new_method,0,"['\r\n', '2', '|', 'MATERIALS', 'AND', 'METHOD...","['O', 'O', 'O', 'O', 'O', 'O', 'O']",\r\n 2 | MATERIALS AND METHODS \r\n,[],0,0,0,0,0,0
1,My_pdf325new_method,1,"['2.1', '|', 'Ethics', 'statement', '\r\n']","['O', 'O', 'O', 'O', 'O']",2.1 | Ethics statement \r\n,[],0,0,0,0,0,0
2,My_pdf325new_method,2,"['Each', 'procedure', 'was', 'conducted', 'in'...","['B-welfare', 'I-welfare', 'I-welfare', 'I-wel...",Each procedure was conducted in accordance wit...,['welfare'],0,1,0,0,0,0
3,My_pdf325new_method,3,"['All', 'efforts', 'were', 'made', 'to', 'mini...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",All efforts were made to minimize the number o...,[],0,0,0,0,0,0
4,My_pdf325new_method,4,"['2.2', '|', 'Animal', 'grouping', 'and', 'mod...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']",2.2 | Animal grouping and model establishment ...,[],0,0,0,0,0,0


In [29]:
sentence_df.shape

(66108, 12)

## Pattern based relevant sentences classification

In [96]:
def evaluate_keyword_detection(df, target_col='age', keyword_col='has_age_kw'):
    """
    Computes and prints TP, FP, FN, precision, recall, and F1 score.
    """
    tp = ((df[keyword_col] == True) & (df[target_col] == 1)).sum()
    fp = ((df[keyword_col] == True) & (df[target_col] == 0)).sum()
    fn = ((df[keyword_col] == False) & (df[target_col] == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    print(f"--- Evaluation for '{keyword_col}' predicting '{target_col}' ---")
    print(f"True Positives (TP): {tp}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print("------------------------------------------------------------")

    return precision, recall, f1

### age

In [99]:
sentence_df_methods = sentence_df[~sentence_df['doc_id'].str.contains('title\^abstract', regex=True)]
sentence_df_methods.shape

(58201, 14)

In [100]:
sentence_df_methods.head()

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,welfare,blinding,randomization,arrive,power,species,has_species_kw
0,My_pdf325new_method,0,"['\r\n', '2', '|', 'MATERIALS', 'AND', 'METHOD...","['O', 'O', 'O', 'O', 'O', 'O', 'O']",\r\n 2 | MATERIALS AND METHODS \r\n,[],0,0,0,0,0,0,species-other,False
1,My_pdf325new_method,1,"['2.1', '|', 'Ethics', 'statement', '\r\n']","['O', 'O', 'O', 'O', 'O']",2.1 | Ethics statement \r\n,[],0,0,0,0,0,0,species-other,False
2,My_pdf325new_method,2,"['Each', 'procedure', 'was', 'conducted', 'in'...","['B-welfare', 'I-welfare', 'I-welfare', 'I-wel...",Each procedure was conducted in accordance wit...,['welfare'],0,1,0,0,0,0,species-other,False
3,My_pdf325new_method,3,"['All', 'efforts', 'were', 'made', 'to', 'mini...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",All efforts were made to minimize the number o...,[],0,0,0,0,0,0,species-other,False
4,My_pdf325new_method,4,"['2.2', '|', 'Animal', 'grouping', 'and', 'mod...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']",2.2 | Animal grouping and model establishment ...,[],0,0,0,0,0,0,species-other,False


In [103]:
# Define the pattern
age_keywords_pattern = (
    r'\b('
    r'age|ages|aged|aging|old|older|young|adult|adults|mature|senescent|'
    r'neonatal|neonate|newborn|pup|pups|juvenile|weanling|weaning|'
    r'postnatal|prenatal|prepubescent|fetal|fetus|fetuses|'
    r'day[-\s]?old|week[-\s]?old|month[-\s]?old|year[-\s]?old|'     # hyphen or space
    r'dayold|weekold|monthold|yearold|'                            # concatenated forms
    r'\d+\s*[-–to]+\s*\d+\s*(day|week|wk|month|year|yr)s?|'           # ranges like 6-8 weeks
    r'\d+\s*(day|week|wk|month|year|yr)s?(?!\s*old)|'                 # standalone like "8 weeks"
    r'after birth|post[-\s]?birth'
    r')\b'
)
# Create the boolean mask
sentence_df_methods['has_age_kw'] = sentence_df_methods['sent_txt'].str.contains(age_keywords_pattern, case=False, regex=True)

print("Rows containing age keywords:", sentence_df_methods.has_age_kw.sum())

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/3932456871.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sentence_df_methods['has_age_kw'] = sentence_df_methods['sent_txt'].str.contains(age_keywords_pattern, case=False, regex=True)


Rows containing age keywords: 3009


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/3932456871.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['has_age_kw'] = sentence_df_methods['sent_txt'].str.contains(age_keywords_pattern, case=False, regex=True)


In [104]:
evaluate_keyword_detection(sentence_df_methods)

--- Evaluation for 'has_age_kw' predicting 'age' ---
True Positives (TP): 448
False Positives (FP): 2561
False Negatives (FN): 24
Precision: 0.15
Recall: 0.95
F1 Score: 0.26
------------------------------------------------------------


(0.14888667331339314, 0.9491525423728814, 0.25739729962654406)

In [43]:
missed_age = sentence_df_methods[(sentence_df_methods['age'] == 1) & (sentence_df_methods['has_age_kw'] == False)]
missed_age.to_csv("data/debug_regex/./data/sentence_classification/missed_age_kw.csv")

In [44]:
missed_age.shape

(24, 13)

In [47]:
df_age_kw = sentence_df_methods[sentence_df_methods['has_age_kw'] == 1]
df_age_kw.to_csv("data/debug_regex/./data/sentence_classification/sent_with_age_kw.csv")
df_age_kw.shape

(3009, 13)

In [48]:
df_age_kw['doc_id'].nunique()

660

In [49]:
sentence_df.head()

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,welfare,blinding,randomization,arrive,power
0,My_pdf325new_method,0,"['\r\n', '2', '|', 'MATERIALS', 'AND', 'METHOD...","['O', 'O', 'O', 'O', 'O', 'O', 'O']",\r\n 2 | MATERIALS AND METHODS \r\n,[],0,0,0,0,0,0
1,My_pdf325new_method,1,"['2.1', '|', 'Ethics', 'statement', '\r\n']","['O', 'O', 'O', 'O', 'O']",2.1 | Ethics statement \r\n,[],0,0,0,0,0,0
2,My_pdf325new_method,2,"['Each', 'procedure', 'was', 'conducted', 'in'...","['B-welfare', 'I-welfare', 'I-welfare', 'I-wel...",Each procedure was conducted in accordance wit...,['welfare'],0,1,0,0,0,0
3,My_pdf325new_method,3,"['All', 'efforts', 'were', 'made', 'to', 'mini...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",All efforts were made to minimize the number o...,[],0,0,0,0,0,0
4,My_pdf325new_method,4,"['2.2', '|', 'Animal', 'grouping', 'and', 'mod...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']",2.2 | Animal grouping and model establishment ...,[],0,0,0,0,0,0


### species

In [53]:
# Define the species patterns
species_patterns = {
    "rat": [r"\brats?\b"],
    "mouse": [r"\bmouse\b", r"\bmice\b"],
    "cat": [r"\bcats?\b"],
    "dog": [r"\bdogs?\b"],
    "guinea pig": [r"\bguinea pig\s?\b", r"\bguinea\b"],
    "monkey": [r"\bmonkeys?\b", r"\bmacaques?\b", r"\bchimpanzees?\b", r"\borangutans?\b", r"\bbonoboss?\b", r"\bgibbons?\b"],
    "pig": [r"\bpigs?\b", r"\bswines?\b", r"\bpiglets?\b"],
    "rabbit": [r"\brabbits?\b"],
    "sheep": [r"\bsheep?\b"],
    "species-other": []  # Fallback
}

# Function to match ent_text to species
def match_species(text):
    for species, patterns in species_patterns.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                return species
    return "species-other"

# Apply to a column like new_df_age['ent_text']
sentence_df['species'] = sentence_df['sent_txt'].apply(match_species)
sentence_df['has_species_kw'] = (sentence_df['species'] != "species-other")

In [55]:
sentence_df

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,welfare,blinding,randomization,arrive,power,species,has_species_kw
0,My_pdf325new_method,0,"['\r\n', '2', '|', 'MATERIALS', 'AND', 'METHOD...","['O', 'O', 'O', 'O', 'O', 'O', 'O']",\r\n 2 | MATERIALS AND METHODS \r\n,[],0,0,0,0,0,0,species-other,False
1,My_pdf325new_method,1,"['2.1', '|', 'Ethics', 'statement', '\r\n']","['O', 'O', 'O', 'O', 'O']",2.1 | Ethics statement \r\n,[],0,0,0,0,0,0,species-other,False
2,My_pdf325new_method,2,"['Each', 'procedure', 'was', 'conducted', 'in'...","['B-welfare', 'I-welfare', 'I-welfare', 'I-wel...",Each procedure was conducted in accordance wit...,['welfare'],0,1,0,0,0,0,species-other,False
3,My_pdf325new_method,3,"['All', 'efforts', 'were', 'made', 'to', 'mini...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",All efforts were made to minimize the number o...,[],0,0,0,0,0,0,species-other,False
4,My_pdf325new_method,4,"['2.2', '|', 'Animal', 'grouping', 'and', 'mod...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']",2.2 | Animal grouping and model establishment ...,[],0,0,0,0,0,0,species-other,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66103,My_pdf96new_method,123,"['If', 'P', '<', '.']","['O', 'O', 'O', 'O']",If P < .,[],0,0,0,0,0,0,species-other,False
66104,My_pdf96new_method,124,"['05', ',', 'F', '-', 'ratios', 'were', 'calcu...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","05 , F - ratios were calculated using the Gree...",[],0,0,0,0,0,0,species-other,False
66105,My_pdf96new_method,125,"['The', 'accepted', 'level', 'of', 'significan...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...",The accepted level of significance for the tes...,[],0,0,0,0,0,0,species-other,False
66106,My_pdf96new_method,126,"['05', '.']","['O', 'O']",05 .,[],0,0,0,0,0,0,species-other,False


In [56]:
print("Rows containing species keywords:", sentence_df.has_species_kw.sum())

Rows containing species keywords: 11554


In [57]:
evaluate_keyword_detection(sentence_df, target_col='age', keyword_col='has_species_kw')

--- Evaluation for 'has_species_kw' predicting 'age' ---
True Positives (TP): 442
False Positives (FP): 11112
False Negatives (FN): 46
Precision: 0.04
Recall: 0.91
------------------------------------------------------------


(0.03825514973169465, 0.9057377049180327)

### welfare

In [107]:
welfare_keywords_pattern = (
    r"\b(approved|approval|authorized|reviewed|"
    r"carried out (in|under)? (strict )?(accordance|compliance|adherence)? (with|to)|"
    r"conducted (in|under)? (strict )?(accordance|compliance|adherence)? (with|to)|"
    r"performed (in|under)? (strict )?(accordance|compliance|adherence)? (with|to)|"
    r"in (strict )?(accordance|compliance|adherence)? (with|to)|"
    r"according to( the)? (institutional |national |international |ethical )?(guidelines|policy|regulations|standards|rules|principles)|"
    r"conformed (to|with)|"
    r"complied (with|to)?( relevant)? (guidelines|regulations|standards)?|"
    r"treated according to( the)? (guidelines|regulations|standards)?|"
    r"in (compliance|accord|agreement|adherence) (with|to)|"
    r"(Institutional|Animal|Ethics) Committee|"
    r"(IACUC|CPCSEA)|"
    r"Guide for the Care and Use of Laboratory Animals|"
    r"Directive (86/609|2010/63)[/\w]*|"
    r"ARRIVE guidelines|"
    r"Declaration of Helsinki|"
    r"ethical (standards|guidelines|policies)|"
    r"international (laws|standards|guidelines)|"
    r"regulations (of|from|by) .*animal.*"
    r")\b"
)

# Create the boolean mask using the correct variable
sentence_df_methods['has_welfare_kw'] = sentence_df_methods['sent_txt'].str.contains(
    welfare_keywords_pattern, case=False, regex=True
)
exclude_pattern = r"\b(human|humans|author[s]?|subject[s]?|patient[s]?|volunteer[s]?|donor[s]?|method described by|" \
                  r"manufacturer(?:'s)?|company|euthanized|inc\b|ltd\b|sigma|fisher|thermo|roche|abbott|bio-rad)\b"

# Overwrite has_welfare_kw: set to False if excluded terms are present
sentence_df_methods.loc[
    sentence_df_methods['sent_txt'].str.contains(exclude_pattern, case=False, regex=True),
    'has_welfare_kw'
] = False

sentence_df_methods.loc[
    sentence_df_methods['sent_txt'].str.split().str.len() <= 4,
    'has_welfare_kw'
] = False

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/2751987042.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sentence_df_methods['has_welfare_kw'] = sentence_df_methods['sent_txt'].str.contains(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/2751987042.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['has_welfare_kw'] = sentence_df_methods['sent_txt'].str.contains(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/2751987042.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sentence_df_methods

In [108]:
print("Rows containing welfare keywords:", sentence_df_methods.has_welfare_kw.sum())

Rows containing welfare keywords: 847


In [109]:
sentence_df_methods.shape

(58201, 16)

In [111]:
sentence_df_methods[sentence_df_methods['has_welfare_kw']==True].welfare.value_counts()


welfare
1    654
0    193
Name: count, dtype: int64

#### NB: grouped is by the document id -> evaluate not each sentence classification, but the final perfromance based on the document level

In [116]:
sentence_df_methods['welfare'] = sentence_df_methods['welfare'].astype(bool)

# Group and aggregate
grouped_df = sentence_df_methods.groupby("doc_id").agg({
    "welfare": "any",
    "has_welfare_kw": "any"
}).reset_index()

# Optionally convert boolean back to int (0 or 1)
grouped_df["welfare"] = grouped_df["welfare"].astype(int)
grouped_df["has_welfare_kw"] = grouped_df["has_welfare_kw"].astype(int)
grouped_df

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/265170817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['welfare'] = sentence_df_methods['welfare'].astype(bool)


,doc_id,welfare,has_welfare_kw
0,My_pdf100new_method,1,1
1,My_pdf101new_method,1,1
2,My_pdf102new_method,1,1
3,My_pdf103new_method,1,1
4,My_pdf104new_method,1,1
...,...,...,...
720,My_pdf995_method,1,1
721,My_pdf997_method,1,1
722,My_pdf998_method,1,1
723,My_pdf99new_method,1,1


In [117]:
evaluate_keyword_detection(grouped_df, target_col='welfare', keyword_col='has_welfare_kw')

--- Evaluation for 'has_welfare_kw' predicting 'welfare' ---
True Positives (TP): 618
False Positives (FP): 13
False Negatives (FN): 23
Precision: 0.98
Recall: 0.96
F1 Score: 0.97
------------------------------------------------------------


(0.9793977812995246, 0.9641185647425897, 0.9716981132075472)

In [118]:
evaluate_keyword_detection(sentence_df_methods, target_col='welfare', keyword_col='has_welfare_kw')

--- Evaluation for 'has_welfare_kw' predicting 'welfare' ---
True Positives (TP): 654
False Positives (FP): 193
False Negatives (FN): 52
Precision: 0.77
Recall: 0.93
F1 Score: 0.84
------------------------------------------------------------


(0.7721369539551358, 0.9263456090651558, 0.8422408242112042)

In [77]:
sentence_df_methods[(sentence_df_methods['welfare'] == 1) & (sentence_df_methods['has_welfare_kw'] == False)].to_csv("data/debug_regex/debug_welfare_fn.csv")
sentence_df_methods[(sentence_df_methods['welfare'] == 0) & (sentence_df_methods['has_welfare_kw'] == True)].to_csv("data/debug_regex/debug_welfare_fp.csv")


### blinding

In [335]:
blinding_keywords_pattern = (
    "blinded|were blind|blind"
)

# Create the boolean mask using the correct variable
sentence_df_methods['has_blinding_kw'] = sentence_df_methods['sent_txt'].str.contains(
    blinding_keywords_pattern, case=False, regex=True
)
print("Rows containing blinding keywords:", sentence_df_methods.has_blinding_kw.sum())

Rows containing blinding keywords: 391


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19283/1742141373.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['has_blinding_kw'] = sentence_df_methods['sent_txt'].str.contains(


In [337]:
evaluate_keyword_detection(sentence_df_methods, target_col='blinding', keyword_col='has_blinding_kw')

--- Evaluation for 'has_blinding_kw' predicting 'blinding' ---
True Positives (TP): 361
False Positives (FP): 30
False Negatives (FN): 28
Precision: 0.92
Recall: 0.93
------------------------------------------------------------


(0.9232736572890026, 0.9280205655526992)

### randomization

In [555]:
randomization_keywords_pattern = (
    r"\b("
    r"randomized(?: control(?:led)? (trial|study))?|"
    r"randomized (mice|rats|animals|groups)|"
    r"(were|was|had been)?\s*randomly (assigned|divided|grouped|grouping|selected|allocated|placed|chosen|culled|distributed)|"
    r"(assigned|divided|grouped|grouping|allocated|chosen|selected|culled) randomly|"
    r"(assigned|divided|grouped|allocated|placed|selected|chosen) at random|"
    r"pseudorandomly (assigned|placed|culled|selected)?|"
    r"random assignment|"
    r"random allocation|"
    r"randomization (was performed|of animals)?|"
    r"randomly allocated|"
    r"animals were randomized to|"
    r"randomized into (groups|treatment groups)|"
    r"randomly selected (mice|rats|subjects|animals)"
    r")\b"
)

# Create the boolean mask using the correct variable
sentence_df_methods['has_random_kw'] = sentence_df_methods['sent_txt'].str.contains(
    randomization_keywords_pattern, case=False, regex=True
)
print("Rows containing randomization keywords:", sentence_df_methods.has_random_kw.sum())

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19283/3331093236.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sentence_df_methods['has_random_kw'] = sentence_df_methods['sent_txt'].str.contains(


Rows containing randomization keywords: 507


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19283/3331093236.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['has_random_kw'] = sentence_df_methods['sent_txt'].str.contains(


In [578]:
sentence_df_methods['randomization'] = sentence_df_methods['randomization'].astype(bool)

# Group and aggregate
grouped_df = sentence_df_methods.groupby("doc_id").agg({
    "randomization": "any",
    "has_random_kw": "any"
}).reset_index()

# Optionally convert boolean back to int (0 or 1)
grouped_df["randomization"] = grouped_df["randomization"].astype(int)
grouped_df["has_random_kw"] = grouped_df["has_random_kw"].astype(int)
grouped_df.head()

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19283/1665136390.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['randomization'] = sentence_df_methods['randomization'].astype(bool)


,doc_id,randomization,has_random_kw
0,My_pdf100new_method,0,0
1,My_pdf101new_method,0,0
2,My_pdf102new_method,0,0
3,My_pdf103new_method,0,0
4,My_pdf104new_method,0,0


In [580]:
evaluate_keyword_detection(grouped_df, target_col='randomization', keyword_col='has_random_kw')

--- Evaluation for 'has_random_kw' predicting 'randomization' ---
True Positives (TP): 298
False Positives (FP): 40
False Negatives (FN): 19
Precision: 0.88
Recall: 0.94
------------------------------------------------------------


(0.8816568047337278, 0.9400630914826499)

In [557]:
evaluate_keyword_detection(sentence_df_methods, target_col='randomization', keyword_col='has_random_kw')

--- Evaluation for 'has_random_kw' predicting 'randomization' ---
True Positives (TP): 375
False Positives (FP): 132
False Negatives (FN): 54
Precision: 0.74
Recall: 0.87
------------------------------------------------------------


(0.7396449704142012, 0.8741258741258742)

In [559]:
sentence_df_methods[(sentence_df_methods['randomization'] == 1) & (sentence_df_methods['has_random_kw'] == False)].to_csv("data/debug_regex/debug_random_fn.csv")
sentence_df_methods[(sentence_df_methods['randomization'] == 0) & (sentence_df_methods['has_random_kw'] == True)].to_csv("data/debug_regex/debug_random_fp.csv")


### arrive

In [346]:
randomizaiton_keywords_pattern = (
    "arrive"
)

# Create the boolean mask using the correct variable
sentence_df_methods['has_arrive_kw'] = sentence_df_methods['sent_txt'].str.contains(
    randomizaiton_keywords_pattern, case=False, regex=True
)
print("Rows containing arrive keywords:", sentence_df_methods.has_arrive_kw.sum())

Rows containing arrive keywords: 20


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19283/3949533471.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['has_arrive_kw'] = sentence_df_methods['sent_txt'].str.contains(


In [348]:
evaluate_keyword_detection(sentence_df_methods, target_col='arrive', keyword_col='has_arrive_kw')

--- Evaluation for 'has_arrive_kw' predicting 'arrive' ---
True Positives (TP): 13
False Positives (FP): 7
False Negatives (FN): 1
Precision: 0.65
Recall: 0.93
------------------------------------------------------------


(0.65, 0.9285714285714286)

### sample size calculation

In [86]:
import re

def compile_patterns():
    # -------- Negative cues --------
    negative_parts = [
        r"\bno\s+(?:a\s+priori\s+)?(?:sample[-\s]?size\s+(?:calculation|estimation|determination)|power\s+analysis)\b",
        r"\b(?:sample[-\s]?size|power\s+analysis)\s+(?:was|were)\s+not\s+(?:performed|conducted|done|carried\s+out|calculated|estimated|determined)\b",
        r"\bno\s+sample[-\s]?size\s+(?:was|were)\s+(?:calculated|estimated|determined|computed)\b",
        r"\b(?:without|lacking)\s+(?:a\s+priori\s+)?(?:sample[-\s]?size\s+(?:calculation|estimation|determination)|power\s+analysis)\b",
        r"\bno\s+(?:power\s+analysis|power\s+calculation)\b",
    ]
    neg_rx = re.compile("|".join(negative_parts), re.IGNORECASE)

    # -------- Positive cues --------
    positive_parts = [
        r"\bsample[-\s]?size(?:s)?\s+(?:was|were)\s+(?:calculated|estimated|determined|computed|predetermined)\b",
        r"\b(?:we|authors?)\s+(?:calculated|estimated|determined)\s+the\s+sample[-\s]?size\b",
        r"\bsample[-\s]?size\s+(?:calculation|estimation|determination)\s+(?:was|were)\s+(?:performed|conducted|done|carried\s+out)\b",

        r"\b(?:power\s+analysis|power\s+calculation)\s+(?:was|were)\s+(?:performed|conducted|done|carried\s+out)\b",
        r"\b(?:power\s+analysis|power\s+calculation)\b",

        r"\b(?:samples?\s+sizes?|number\s+of\s+(?:mice|rats|animals|subjects)(?:\s+per\s+group)?)\s+was\s+(?:determined|set|chosen)\s+(?:by|based\s+on|according\s+to)\b",

        r"\bpower\s*(?:=|of)\s*\d+(?:\.\d+)?\s*(?:%|percent)?\b.*?\b(?:alpha|α)\s*(?:=|of)?\s*0?\.\d+\b",
        r"\b(?:alpha|α)\s*(?:=|of)?\s*0?\.\d+\b.*?\bpower\s*(?:=|of)\s*\d+(?:\.\d+)?\s*(?:%|percent)?\b",

        r"\bensure\s+adequate\s+power\b",
        r"\bsufficient\s+to\s+attain\s+statistical\s+power\b",
    ]
    pos_rx = re.compile("|".join(positive_parts), re.IGNORECASE)

    return neg_rx, pos_rx


neg_rx, pos_rx = compile_patterns()

# -------- Final boolean: has_power_kw --------
sentence_df_methods["has_power_kw"] = (
    sentence_df_methods["sent_txt"].str.contains(pos_rx, na=False)
    & ~sentence_df_methods["sent_txt"].str.contains(neg_rx, na=False)
)

print("Rows containing power/sample size (positive only):",
      sentence_df_methods["has_power_kw"].sum())

Rows containing power/sample size (positive only): 16


/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/710962443.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods["has_power_kw"] = (


In [90]:
sentence_df_methods['power'] = sentence_df_methods['power'].astype(bool)

# Group and aggregate
grouped_df = sentence_df_methods.groupby("doc_id").agg({
    "power": "any",
    "has_power_kw": "any"
}).reset_index()

# Optionally convert boolean back to int (0 or 1)
grouped_df["power"] = grouped_df["power"].astype(int)
grouped_df["has_power_kw"] = grouped_df["has_power_kw"].astype(int)
grouped_df.head()

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70975/2494995199.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sentence_df_methods['power'] = sentence_df_methods['power'].astype(bool)


,doc_id,power,has_power_kw
0,My_pdf100new_method,0,0
1,My_pdf101new_method,0,0
2,My_pdf102new_method,0,0
3,My_pdf103new_method,0,0
4,My_pdf104new_method,0,0


In [88]:
evaluate_keyword_detection(sentence_df_methods, target_col='power', keyword_col='has_power_kw')

--- Evaluation for 'has_power_kw' predicting 'power' ---
True Positives (TP): 10
False Positives (FP): 6
False Negatives (FN): 15
Precision: 0.62
Recall: 0.40
------------------------------------------------------------


(0.625, 0.4)

In [92]:
evaluate_keyword_detection(grouped_df, target_col='power', keyword_col='has_power_kw')

--- Evaluation for 'has_power_kw' predicting 'power' ---
True Positives (TP): 11
False Positives (FP): 2
False Negatives (FN): 10
Precision: 0.85
Recall: 0.52
------------------------------------------------------------


(0.8461538461538461, 0.5238095238095238)